# Entrega 1: coleta das bases brutas

Único objetivo deste notebook: **trazer os arquivos originais para o disco**. Nada é lido, transformado ou filtrado aqui. O tratamento e a delimitação por Minas Gerais ficam na entrega 2.

| Fonte | O que é coletado |
|---|---|
| CVM | Demonstrações Financeiras Padronizadas, um ZIP por ano, mais o cadastro das companhias abertas |
| BACEN, IF.data | Catálogo, cadastro, dicionário de campos e valores dos conglomerados prudenciais, dezembros de 2014 a 2025 |
| BACEN, resolução | Relação oficial de instituições em liquidação extrajudicial, intervenção e RAET |
| Receita Federal | Os 37 arquivos de dados públicos de CNPJ da competência escolhida |

Tudo é retomável: o que já está em disco não é baixado de novo. Pode interromper e rodar outra vez sem perder nada.

## 1. Configuração

In [1]:
!pip install -q pandas numpy pyarrow requests tqdm

In [2]:
import re
import io
import time
import json
import zipfile
import warnings
import unicodedata
from pathlib import Path
from datetime import datetime
from urllib.parse import unquote
from xml.etree import ElementTree

import pandas as pd
import requests
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

In [3]:
RAIZ = Path(r"G:\Meu Drive\TCC\Damaris Rebeca")

DIR_BRUTO = RAIZ / "01_bruto"
DIR_CVM = DIR_BRUTO / "cvm"
DIR_IFDATA = DIR_BRUTO / "bcb_ifdata"
DIR_RES = DIR_BRUTO / "bcb_resolucao"
DIR_RFB = DIR_BRUTO / "rfb"

for d in (DIR_BRUTO, DIR_CVM, DIR_IFDATA, DIR_RES, DIR_RFB):
    d.mkdir(parents=True, exist_ok=True)

ANO_INICIAL_CVM = 2011
ANO_FINAL_CVM = datetime.now().year

IFDATA_TIPOS = [1004, 1009]
IFDATA_ANOS = range(2014, 2026)

RFB_COMPETENCIA = "2026-08"
RFB_TOKEN = "YggdBLfdninEJX9"

print("Raiz:", RAIZ, "| existe:", RAIZ.exists())
print("CVM:", ANO_INICIAL_CVM, "a", ANO_FINAL_CVM)
print("IF.data: dezembros de", min(IFDATA_ANOS), "a", max(IFDATA_ANOS), "| tipos", IFDATA_TIPOS)
print("Receita Federal: competencia", RFB_COMPETENCIA)

Raiz: G:\Meu Drive\TCC\Damaris Rebeca | existe: True
CVM: 2011 a 2026
IF.data: dezembros de 2014 a 2025 | tipos [1004, 1009]
Receita Federal: competencia 2026-08


In [5]:
SESSAO = requests.Session()
SESSAO.headers.update({"User-Agent": "Mozilla/5.0 (pesquisa academica)"})

def mb(n):
    return f"{n/1e6:,.1f} MB"

def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    return " ".join(s.upper().split())

def baixar(url, destino, forcar=False, tentativas=4, timeout=600, auth=None, silencioso=False):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists() and destino.stat().st_size > 0 and not forcar:
        return destino
    for t in range(1, tentativas + 1):
        try:
            with SESSAO.get(url, stream=True, timeout=timeout, auth=auth) as r:
                if r.status_code == 404:
                    return None
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0))
                parcial = destino.with_name(destino.name + ".parcial")
                barra = tqdm(total=total, unit="B", unit_scale=True, leave=False,
                             desc=destino.name[:40], disable=silencioso)
                with open(parcial, "wb") as f:
                    for bloco in r.iter_content(chunk_size=1 << 20):
                        f.write(bloco)
                        barra.update(len(bloco))
                barra.close()
                parcial.replace(destino)
            return destino
        except Exception as e:
            if t == tentativas:
                print(f"FALHA {destino.name}: {type(e).__name__}: {str(e)[:80]}")
                return None
            time.sleep(3 * t)
    return None

def pegar_json(url, timeout=180, tentativas=3):
    for t in range(1, tentativas + 1):
        try:
            r = SESSAO.get(url, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception:
            if t == tentativas:
                return None
            time.sleep(2 * t)
    return None

def diagnosticar_zip(caminho):
    caminho = Path(caminho)
    if not caminho.exists():
        return "ausente"
    if caminho.stat().st_size == 0:
        return "vazio"
    if not zipfile.is_zipfile(caminho):
        return "corrompido"
    try:
        with zipfile.ZipFile(caminho) as z:
            if not [m for m in z.namelist() if not m.endswith("/")]:
                return "sem membros"
    except Exception:
        return "corrompido"
    return "ok"

---
## 2. CVM, companhias abertas

Um ZIP por ano com todos os demonstrativos padronizados, mais o cadastro. O cadastro é rebaixado sempre: é pequeno, muda toda semana e é dele que sai a situação de cada companhia, incluindo os carimbos de recuperação judicial e liquidação usados como rótulo na entrega 2.

In [9]:
CVM_DFP = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ano}.zip"
CVM_CAD = "https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/cad_cia_aberta.csv"

anos_cvm = list(range(ANO_INICIAL_CVM, ANO_FINAL_CVM + 1))
faltando = [a for a in anos_cvm if diagnosticar_zip(DIR_CVM / f"dfp_{a}.zip") != "ok"]

print(f"{len(anos_cvm) - len(faltando)} anos ja em disco | {len(faltando)} a baixar: {faltando}")

for ano in tqdm(faltando, desc="DFP"):
    alvo = DIR_CVM / f"dfp_{ano}.zip"
    if alvo.exists():
        alvo.unlink()
    baixar(CVM_DFP.format(ano=ano), alvo)

baixar(CVM_CAD, DIR_CVM / "cad_cia_aberta.csv", forcar=True)

estado_cvm = {p.name: diagnosticar_zip(p) for p in sorted(DIR_CVM.glob("*.zip"))}
print("\nintegridade:", pd.Series(estado_cvm).value_counts().to_dict())
print("problemas:", {k: v for k, v in estado_cvm.items() if v != "ok"} or "nenhum")
print("cadastro:", mb((DIR_CVM / "cad_cia_aberta.csv").stat().st_size))

16 anos ja em disco | 0 a baixar: []


DFP: 0it [00:00, ?it/s]

cad_cia_aberta.csv:   0%|          | 0.00/1.49M [00:00<?, ?B/s]


integridade: {'ok': 16}
problemas: nenhum
cadastro: 1.5 MB


---
## 3. BACEN, IF.data

O IF.data tem API REST própria em `www3.bcb.gov.br/ifdata/rest`, independente do Olinda. São três chamadas: o catálogo, que lista tudo o que existe, e o endpoint de arquivos, que entrega cada JSON.

**Recorte adotado.** Tipo de instituição 1004 (Conglomerados Prudenciais) até 2022 e 1009 (mesmo recorte, identificador novo) de 2023 em diante. São contíguos e formam uma série única de 2014 a 2025. O tipo 1005, de conglomerados financeiros, iria até 2000 mas perde o Índice de Basileia a partir de 2015, que é o indicador de solvência central para instituição financeira.

Por data-base são coletados quatro conjuntos: `cadastro` (as instituições, com UF e segmento), `info` (o dicionário que traduz o id de cada campo), `trel` (a definição de colunas de cada relatório) e `dados` (os valores). Os `dados` não são separados por relatório, então todos precisam vir.

In [12]:
IFDATA_REST = "https://www3.bcb.gov.br/ifdata/rest"

def ifdata_catalogo(nome):
    destino = DIR_IFDATA / f"catalogo_{nome}.json"
    baixar(f"{IFDATA_REST}/{nome}", destino, forcar=not destino.exists(), timeout=600)
    if destino.exists() and destino.stat().st_size > 0:
        return json.loads(destino.read_text(encoding="utf-8"))
    return []

catalogo = sorted(ifdata_catalogo("relatorios2000a2024") + ifdata_catalogo("relatorios2025a2030"),
                  key=lambda x: x["dt"])
datas = [x["dt"] for x in catalogo]
print(f"{len(datas)} datas-base no catalogo, de {datas[0]} a {datas[-1]}")

RELATORIOS_ALVO = {normalizar(x) for x in
                   ["Resumo", "Ativo", "Passivo", "Demonstracao de Resultado",
                    "Informacoes de Capital", "Segmentacao"]}

DATAS_ALVO = [d for d in datas if str(d).endswith("12") and int(str(d)[:4]) in IFDATA_ANOS]
print(f"{len(DATAS_ALVO)} datas-base alvo:", DATAS_ALVO)

106 datas-base no catalogo, de 200003 a 202606
12 datas-base alvo: [201412, 201512, 201612, 201712, 201812, 201912, 202012, 202112, 202212, 202312, 202412, 202512]


In [14]:
plano = []
for dt in DATAS_ALVO:
    bloco = next((x for x in catalogo if x["dt"] == dt), None)
    if bloco is None:
        print(f"{dt} ausente no catalogo")
        continue
    for f in bloco["files"]:
        nome = f["f"].split("/")[-1]
        manter = False
        if nome.startswith(("dados", "sel", "info")):
            manter = True
        elif nome.startswith("cadastro"):
            manter = any(nome.endswith(f"_{t}.json") for t in IFDATA_TIPOS)
        elif nome.startswith("trel") and f.get("trel"):
            t = f["trel"]
            manter = (normalizar(t["n"]) in RELATORIOS_ALVO
                      and any(s["id"] in IFDATA_TIPOS for s in t.get("s", [])))
        if manter:
            plano.append({"dt": dt, "grupo": re.sub(r"\d.*", "", nome), "arquivo": f["f"]})

plano = pd.DataFrame(plano).drop_duplicates(subset=["arquivo"])
print(f"{len(plano)} arquivos planejados em {plano['dt'].nunique()} datas-base\n")
print(plano.groupby("grupo").size().to_string())

153 arquivos planejados em 12 datas-base

grupo
cadastro    12
dados       49
info        12
sel         12
trel        68


In [16]:
def baixar_ifdata(caminho_remoto):
    partes = caminho_remoto.split("/")
    destino = DIR_IFDATA / partes[-2] / partes[-1]
    if destino.exists() and destino.stat().st_size > 0:
        return destino, True
    return baixar(f"{IFDATA_REST}/arquivos?nomeArquivo={caminho_remoto}", destino,
                  timeout=900, silencioso=True), False

novos = cache = falhas = 0
for caminho in tqdm(plano["arquivo"].tolist(), desc="IF.data"):
    resultado, cacheado = baixar_ifdata(caminho)
    if resultado is None:
        falhas += 1
        print("falhou:", caminho)
    elif cacheado:
        cache += 1
    else:
        novos += 1

print(f"\nbaixados agora: {novos} | ja existiam: {cache} | falhas: {falhas}")
print("total em disco:", mb(sum(p.stat().st_size for p in DIR_IFDATA.rglob('*.json'))))

IF.data:   0%|          | 0/153 [00:00<?, ?it/s]


baixados agora: 12 | ja existiam: 141 | falhas: 0
total em disco: 236.7 MB


### 3.1 Regimes de resolução

Relação oficial das instituições submetidas a liquidação extrajudicial, intervenção e RAET. É a variável resposta da camada do BACEN. Os recursos são localizados pela API do portal de dados abertos, porque o BACEN republica esses arquivos com identificadores novos.

In [18]:
CKAN = "https://dadosabertos.bcb.gov.br/api/3/action/package_show?id=instituicoes-submetidas-a-regimes-de-resolucao"

pacote = pegar_json(CKAN)
recursos = []
if pacote and pacote.get("success"):
    for r in pacote["result"].get("resources", []):
        recursos.append({"nome": r.get("name"), "formato": (r.get("format") or "").upper(), "url": r.get("url")})

extensoes = {"CSV": ".csv", "XLSX": ".xlsx", "XLS": ".xls", "JSON": ".json"}
baixados = []
for r in recursos:
    if r["formato"] not in extensoes:
        continue
    nome = re.sub(r"[^A-Za-z0-9]+", "_", normalizar(r["nome"]))[:60]
    destino = DIR_RES / (nome + extensoes[r["formato"]])
    if baixar(r["url"], destino, forcar=True, silencioso=True):
        baixados.append(destino)

print(f"{len(recursos)} recursos no conjunto | {len(baixados)} baixados")
for p in baixados:
    print(f"   {p.name:<60} {mb(p.stat().st_size):>12}")

5 recursos no conjunto | 1 baixados
   INSTITUICOES_EM_LIQUIDACAO_EXTRAJUDICIAL.json                      0.0 MB


---
## 4. Receita Federal, dados públicos de CNPJ

Desde o início de 2026 a Receita distribui esses arquivos por uma instância Nextcloud, acessada por WebDAV. O identificador do compartilhamento fica isolado numa constante, porque é o ponto que quebra se a Receita regerar o link público.

A célula confere a integridade de cada ZIP abrindo o índice interno e só baixa o que estiver faltando ou quebrado. Os 7,7 GB já em disco não são tocados.

In [20]:
RFB_HOST = "https://arquivos.receitafederal.gov.br"
RFB_WEBDAV = f"{RFB_HOST}/public.php/webdav"
RFB_DAV = f"{RFB_HOST}/public.php/dav/files/{RFB_TOKEN}"

def listar_webdav(caminho=""):
    url = RFB_WEBDAV + "/" + caminho.strip("/")
    if not url.endswith("/"):
        url += "/"
    for t in range(1, 4):
        try:
            r = SESSAO.request("PROPFIND", url, auth=(RFB_TOKEN, ""),
                               headers={"Depth": "1"}, timeout=120)
            r.raise_for_status()
            arvore = ElementTree.fromstring(r.content)
            itens = []
            for no in arvore.findall("{DAV:}response"):
                href = no.find("{DAV:}href")
                if href is not None and href.text:
                    nome = unquote(href.text).rstrip("/").split("/")[-1]
                    if nome and nome not in ("webdav", caminho.strip("/")):
                        itens.append(nome)
            return sorted(set(itens))
        except Exception as e:
            if t == 3:
                print(f"FALHA PROPFIND: {type(e).__name__}")
                return []
            time.sleep(3 * t)
    return []

competencias = sorted(i for i in listar_webdav() if re.fullmatch(r"20\d{2}-\d{2}", i))
print("competencias publicadas:", competencias)
print("competencia adotada:", RFB_COMPETENCIA,
      "(disponivel)" if RFB_COMPETENCIA in competencias else "(NAO DISPONIVEL)")

no_servidor = [a for a in listar_webdav(RFB_COMPETENCIA) if a.lower().endswith(".zip")]
print(f"{len(no_servidor)} arquivos ZIP no servidor")

competencias publicadas: ['2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10', '2023-11', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08']
competencia adotada: 2026-08 (disponivel)
37 arquivos ZIP no servidor


In [21]:
estado_rfb = {a: diagnosticar_zip(DIR_RFB / a) for a in no_servidor}
problemas = [a for a, s in estado_rfb.items() if s != "ok"]

print("integridade local:", pd.Series(estado_rfb).value_counts().to_dict())
print(f"{len(problemas)} arquivos a (re)baixar:", problemas or "nenhum")

for arq in tqdm(problemas, desc="RFB"):
    destino = DIR_RFB / arq
    if destino.exists():
        destino.unlink()
    if baixar(f"{RFB_DAV}/{RFB_COMPETENCIA}/{arq}", destino) is None:
        baixar(f"{RFB_WEBDAV}/{RFB_COMPETENCIA}/{arq}", destino, auth=(RFB_TOKEN, ""))
    print(f"   {arq:<28} {diagnosticar_zip(destino)}")

estado_rfb = {a: diagnosticar_zip(DIR_RFB / a) for a in no_servidor}
print("\nintegridade final:", pd.Series(estado_rfb).value_counts().to_dict())

integridade local: {'ok': 37}
0 arquivos a (re)baixar: nenhum


RFB: 0it [00:00, ?it/s]


integridade final: {'ok': 37}


---
## 5. Inventário e manifesto

Fecha a entrega 1. O manifesto grava competência, período e contagens, que é o que vai para a seção de metodologia do trabalho como registro da coleta.

In [23]:
def resumir(rotulo, pasta, padrao="*"):
    arquivos = [p for p in pasta.rglob(padrao) if p.is_file()]
    total = sum(p.stat().st_size for p in arquivos)
    print(f"{rotulo:<22}{len(arquivos):>7} arquivos {mb(total):>14}")
    return len(arquivos), total

print(f"{'fonte':<22}{'qtd':>7}{'':>10}{'tamanho':>14}")
n_cvm, b_cvm = resumir("CVM", DIR_CVM)
n_if, b_if = resumir("BACEN IF.data", DIR_IFDATA)
n_res, b_res = resumir("BACEN resolucao", DIR_RES)
n_rfb, b_rfb = resumir("Receita Federal", DIR_RFB, "*.zip")

datas_ifdata = sorted({p.parent.name for p in DIR_IFDATA.rglob("*.json") if p.parent != DIR_IFDATA})
print("\ndatas-base IF.data em disco:", datas_ifdata)
print("faltando:", [str(d) for d in DATAS_ALVO if str(d) not in datas_ifdata] or "nenhuma")

manifesto = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "raiz": str(RAIZ),
    "cvm": {"arquivos": n_cvm, "bytes": b_cvm,
            "anos": sorted(int(re.findall(r"(\d{4})", p.stem)[-1]) for p in DIR_CVM.glob("dfp_*.zip"))},
    "bacen_ifdata": {"arquivos": n_if, "bytes": b_if, "tipos": IFDATA_TIPOS,
                     "datas_base": datas_ifdata},
    "bacen_resolucao": {"arquivos": n_res, "bytes": b_res},
    "receita_federal": {"competencia": RFB_COMPETENCIA, "arquivos": n_rfb, "bytes": b_rfb,
                        "com_problema": {k: v for k, v in estado_rfb.items() if v != "ok"}},
}

(DIR_BRUTO / "manifesto_coleta.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n" + json.dumps(manifesto, ensure_ascii=False, indent=2))

fonte                     qtd                 tamanho
CVM                        17 arquivos       167.9 MB
BACEN IF.data             156 arquivos       236.7 MB
BACEN resolucao             1 arquivos         0.0 MB
Receita Federal            37 arquivos     7,692.1 MB

datas-base IF.data em disco: ['201412', '201512', '201612', '201712', '201812', '201912', '202012', '202112', '202212', '202312', '202412', '202512', 'resolucao']
faltando: nenhuma

{
  "gerado_em": "2026-09-09T20:09:19",
  "raiz": "G:\\Meu Drive\\TCC\\Damaris Rebeca",
  "cvm": {
    "arquivos": 17,
    "bytes": 167866996,
    "anos": [
      2011,
      2012,
      2013,
      2014,
      2015,
      2016,
      2017,
      2018,
      2019,
      2020,
      2021,
      2022,
      2023,
      2024,
      2025,
      2026
    ]
  },
  "bacen_ifdata": {
    "arquivos": 156,
    "bytes": 236692276,
    "tipos": [
      1004,
      1009
    ],
    "datas_base": [
      "201412",
      "201512",
      "201612",
      "201